In [30]:
import pandas as pd
import numpy as np

matches = pd.read_csv('matches.csv')

if 'id' in matches.columns:
    matches = matches.rename(columns={'id': 'match_id'})

cols_to_drop = ['match_type', 'player_of_match', 'target_runs', 'target_overs',
                'super_over', 'umpire1', 'umpire2', 'season', 'city', 'date',
                'toss_winner', 'toss_decision', 'result_margin', 'result', 'method',
                'team1', 'team2']
matches = matches.drop(columns=cols_to_drop, errors='ignore')  # Avoid error if columns missing

team_mapping = {
    "Royal Challengers Bengaluru": "Royal Challengers Bangalore",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings"
}
matches['winner'] = matches['winner'].replace(team_mapping)

venue_mapping = {
    "M Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association Stadium, Mohali",
    "Wankhede Stadium": "Wankhede Stadium",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",
    "Eden Gardens": "Eden Gardens",
    "Eden Gardens, Kolkata": "Eden Gardens",
    "Sawai Mansingh Stadium": "Sawai Mansingh Stadium",
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",
    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Stadium, Hyderabad",
    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium": "MA Chidambaram Stadium, Chepauk",
    "Dr DY Patil Sports Academy": "Dr DY Patil Sports Academy",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",
    "Brabourne Stadium": "Brabourne Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",
    "Himachal Pradesh Cricket Association Stadium": "Himachal Pradesh Cricket Association Stadium",
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Subrata Roy Sahara Stadium": "Subrata Roy Sahara Stadium",
    "Maharashtra Cricket Association Stadium": "Maharashtra Cricket Association Stadium",
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",
    "Arun Jaitley Stadium": "Arun Jaitley Stadium, Delhi",
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium, Delhi",
    "Feroz Shah Kotla": "Arun Jaitley Stadium, Delhi"
}

matches['venue_canonical'] = matches['venue'].map(venue_mapping).fillna(matches['venue'])

matches = matches[['match_id', 'winner', 'venue_canonical']]

deliveries = pd.read_csv('deliveries.csv')

if 'id' in deliveries.columns:
    deliveries = deliveries.rename(columns={'id': 'match_id'})

cols_deliveries = ['match_id', 'inning', 'batting_team', 'bowling_team',
                   'over', 'ball', 'total_runs', 'is_wicket']
deliveries_subset = deliveries[cols_deliveries].copy()

for col in ['batting_team', 'bowling_team']:
    deliveries_subset[col] = deliveries_subset[col].replace(team_mapping)

deliveries_subset['cum_runs'] = deliveries_subset.groupby(['match_id', 'inning'])['total_runs'].cumsum()
deliveries_subset['cum_wickets'] = deliveries_subset.groupby(['match_id', 'inning'])['is_wicket'].cumsum()
deliveries_subset['overs_completed'] = deliveries_subset['over'] + (deliveries_subset['ball'] - 1) / 6
deliveries_subset['current_run_rate'] = np.where(
    deliveries_subset['overs_completed'] == 0,
    0,
    deliveries_subset['cum_runs'] / deliveries_subset['overs_completed']
)


first_innings = deliveries_subset[deliveries_subset['inning'] == 1]
first_innings_final = first_innings.groupby('match_id')['cum_runs'].max().reset_index()
first_innings_final = first_innings_final.rename(columns={'cum_runs': 'first_innings_score'})
first_innings_final['target'] = first_innings_final['first_innings_score'] + 1

final_data = pd.merge(deliveries_subset, matches, on='match_id', how='left')

final_data = pd.merge(final_data, first_innings_final[['match_id', 'target']], on='match_id', how='left')

remaining_overs = 20 - final_data['overs_completed']
final_data['required_run_rate'] = np.where(
    (final_data['inning'] == 2) & (remaining_overs > 0),
    (final_data['target'] - final_data['cum_runs']) / remaining_overs,
    0
)
final_data['required_run_rate'] = final_data['required_run_rate'].replace([np.inf, -np.inf], 0)

final_data['win'] = (final_data['batting_team'] == final_data['winner']).astype(int)

final_data = final_data[final_data['inning'] == 2].copy()

keep_cols = ['match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
             'required_run_rate', 'target', 'batting_team', 'bowling_team', 'venue_canonical', 'win']
final_data = final_data[keep_cols]

final_data.info()
final_data.head()


<class 'pandas.core.frame.DataFrame'>
Index: 125741 entries, 124 to 260919
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   match_id           125741 non-null  int64  
 1   inning             125741 non-null  int64  
 2   cum_runs           125741 non-null  int64  
 3   cum_wickets        125741 non-null  int64  
 4   current_run_rate   125741 non-null  float64
 5   required_run_rate  125741 non-null  float64
 6   target             125741 non-null  int64  
 7   batting_team       125741 non-null  object 
 8   bowling_team       125741 non-null  object 
 9   venue_canonical    125741 non-null  object 
 10  win                125741 non-null  int64  
dtypes: float64(2), int64(6), object(3)
memory usage: 11.5+ MB


,match_id,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team,bowling_team,venue_canonical,win
124,335982,2,1,0,0.0,11.100000,223,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
125,335982,2,2,0,12.0,11.142857,223,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
126,335982,2,2,0,6.0,11.237288,223,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
127,335982,2,3,0,6.0,11.282051,223,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
128,335982,2,4,0,6.0,11.327586,223,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
